# Shor's Algorithm

Factors integers using quantum order-finding.
This notebook demonstrates the full pipeline for factoring 15.

In [ ]:
import cirq
import math
import numpy as np
from fractions import Fraction

## Step 1: Choose a coprime a

In [ ]:
N = 15
a = 7
print(f"N = {N}, a = {a}, gcd({a}, {N}) = {math.gcd(a, N)}")

## Step 2: Find order r of a mod N (classical)

In [ ]:
def find_order(a, N):
    x = 1
    for r in range(1, N + 1):
        x = (x * a) % N
        x = (x * a) % N
        if x == 1:
            return r
    return None

r = find_order(a, N)
print(f"Order r = {r}")
print(f"Verify: {a}^{r} mod {N} = {pow(a, r, N)}")

## Step 3: Factor from order

In [ ]:
half = pow(a, r // 2, N)
f1 = math.gcd(half + 1, N)
f2 = math.gcd(half - 1, N)
print(f"a^(r/2) mod N = {half}")
print(f"gcd({half}+1, {N}) = {f1}")
print(f"gcd({half}-1, {N}) = {f2}")
print(f"\nFactors: {f1} x {f2} = {f1 * f2}")

## Quantum Order-Finding Circuit

The quantum part uses controlled modular exponentiation + inverse QFT
on precision qubits to estimate the phase theta = k/r.

In [ ]:
n_count = 4
n_work = 4
precision = cirq.LineQubit.range(n_count)
work = cirq.LineQubit.range(n_count, n_count + n_work)

circuit = cirq.Circuit(
    cirq.X(work[0]),
    [cirq.H(q) for q in precision],
)
print(f"Circuit: {n_count} precision + {n_work} work qubits")
print(circuit)
print("\n(Full controlled modular exponentiation omitted for simplicity)")

## Trial: Factor all N < 30

In [ ]:
def shor_factor(N):
    if N % 2 == 0:
        return (2, N // 2)
    for a in range(2, N):
        if math.gcd(a, N) != 1:
            continue
        x = 1
        r = 0
        for _ in range(1, N + 1):
            x = (x * a) % N
            r += 1
            if x == 1:
                break
        if r % 2 != 0:
            continue
        half = pow(a, r // 2, N)
        f1 = math.gcd(half + 1, N)
        f2 = math.gcd(half - 1, N)
        if 1 < f1 < N:
            return (f1, N // f1)
        if 1 < f2 < N:
            return (f2, N // f2)
    return None

for n in range(3, 30, 2):
    factors = shor_factor(n)
    if factors:
        print(f"  {n} = {factors[0]} x {factors[1]}")